# Setup

In [1]:
import numpy as np 
import matplotlib.pyplot as plt 
import os 
import yaml 
import json 
import zipfile 
import pandas as pd 
import platform 
from glob import glob 
from pathlib import Path 

WORKING = Path("/kaggle/working/") if Path("/kaggle/working").exists() else Path.cwd()
WORKING.mkdir(parents=True, exist_ok=True)
print("""
     ____ 
    o8%8888,     
   o88%8888888.    
   8'-  -:8888b   
  8'     8888 
  d8.-=. ,==-.:888b  
  >8 `~` :`~' d8888   SWE CODING AGENT // VERSION 1
  88     ,88888   Find Issue -> Bug Fix -> Patch Crafter -> Test Scout 
  88b. `-~ ':88888   Your Code should be as beautiful as mona lisa
  888b ~==~ .:88888  
  88888o--:':::8888   
  `88888| :::' 8888b 
  8888^^'    8888b 
 d888      ,%888b.  
 d88%      %%%8--'-. 
/88:.__ ,    _%-' --- - 
  '''::===..-'  = --.

""")

print(f"Python: {platform.python_version()} | artifact_directory: {WORKING}")
print("Setup Complete!")


     ____ 
    o8%8888,     
   o88%8888888.    
   8'-  -:8888b   
  8'     8888 
  d8.-=. ,==-.:888b  
  >8 `~` :`~' d8888   SWE CODING AGENT // VERSION 1
  88     ,88888   Find Issue -> Bug Fix -> Patch Crafter -> Test Scout 
  88b. `-~ ':88888   Your Code should be as beautiful as mona lisa
  888b ~==~ .:88888  
  88888o--:':::8888   
  `88888| :::' 8888b 
  8888^^'    8888b 
 d888      ,%888b.  
 d88%      %%%8--'-. 
/88:.__ ,    _%-' --- - 
  '''::===..-'  = --.


Python: 3.12.13 | artifact_directory: /kaggle/working
Setup Complete!


# Explore tasks.jsonl

In [2]:
print("=" * 60)
print("Tasks Information")
print("=" * 60)


def _comp_dir() -> Path:
  """Find the Kaggle competition directory."""
  for path in (
    "/kaggle/input/gemma-4-developer-agent",
    "/kaggle/input/competitions/gemma-4-developer-agent",
  ):
    path = Path(path)

    if path.is_dir():
      return path

  raise FileNotFoundError("Competition directory doesn't exist")


# Locate competition directory
comp_dir = _comp_dir()

# Locate tasks.jsonl
task_path = comp_dir / "tasks.jsonl"

if not task_path.exists():
  raise FileNotFoundError(f"{task_path} doesn't exist")


# Load JSONL
task_list = pd.read_json(task_path, lines=True)

if task_list.empty:
  raise ValueError(f"{task_path} is empty")


# Basic validation
assert task_list["instance_id"].is_unique, "Duplicate instance IDs found"


# Create useful EDA columns
tasks = pd.DataFrame({
  "id": task_list["instance_id"],
  "repository": task_list["repo"],
  "issue_chars": task_list["problem_statement"]
    .fillna("")
    .str.len(),

  "reference_patch_bytes": task_list["patch"]
    .fillna("")
    .str.encode("utf-8")
    .str.len(),
})


# Basic information
print(f"Source    : {task_path}")
print(f"Public Tasks : {len(tasks)}")
print(f"Repositories : {tasks['repository'].nunique()}")
print(f"Shape     : {task_list.shape}")

print()


# Repository-level summary
display(
  tasks.groupby("repository")
  .agg(
    tasks=("id", "size"),
    median_issue_chars=("issue_chars", "median"),
    median_reference_patch_bytes=("reference_patch_bytes", "median"),
  )
  .sort_values("tasks", ascending=False)
)

Tasks Information
Source    : /kaggle/input/competitions/gemma-4-developer-agent/tasks.jsonl
Public Tasks : 129
Repositories : 4
Shape     : (129, 8)



,tasks,median_issue_chars,median_reference_patch_bytes
repository,,,
fastapi/fastapi,67,677.0,1576.0
Textualize/rich,48,138.5,1005.0
psf/requests,13,406.0,604.0
encode/httpx,1,243.0,9385.0


# Build The Agent

In [3]:
ROR1_YAML = """\
name: ror-1
model: gemma-4-31b-it-qat-w4a16-ct
description: Evidence-driven software engineering agent for repository issue diagnosis, minimal fixes, targeted verification, and clean patch submission.

instruction: !include prompts/system.md

tools:
 - run_command
 - read_file
 - edit_file
 - write_file
 - get_status
 - submit_patch
 - get_code_neighbors
 - search_similar_code
 - get_code_subgraph

generate_content_config: !include configs/sampling.yaml
"""

ROR1_SAMPLING = """\
temprature: 0.3
max_output_tokens: 16384
seed: 2026

thinking_config:
 thinking_level: HIGH
 thinking_budget: 4096 
 include_thoughts: True
"""

ROR1_PROMPT = """
You are Ror-1.1, an autonomous software engineer solving one repository issue in /workspace. 
Your output is the actual git patch, not a proposed plan. The evaluator applies that patch to a fresh 
checkout and runs separate tests. Finish by calling submit_patch() after inspecting the diff.

MISSION:
resolve the behavior requested by the issue with the smallest complete implementation. Spend the available
budget on evidence and a working patch. Never claim a test passed unless a tool showed that result. 

1.Decode the issue. Extract the requested behavior, affected API, edge cases, and any literal error text. 
Some reports are copied from pull-request templates: ignore boilerplate about discussions, checklists, 
and release process; use the substantive title and description. A terse title is still an actionable 
specification. If it names a method, exception, or symptom, find the implementation and infer a concrete 
failing case from current code. Do not stop merely because no test or hint was supplied. 

2. Locate efficiently. The initial user message includes a repository listing. Start with the named module 
or symbol if present. Otherwise use one bounded `rg -n` search for exact terms, then search nearby API 
names or tests if the first route fails. If `rg` is unavailable, use bounded `grep -R`. Read focused file
ranges, not entire large files. Use code-graph tools only when the graph exists and a real symbol name is
known. `search_similar_code` resolves symbols, not natural-language queries. If a graph call errors, 
switch immediately to source search. Search both implementation and the nearest relevant tests or 
examples, especially public behavior and existing conventions.

3. Form a testable hypothesis. Trace inputs through the affected function and its callers until you can 
explain why the current code fails the requested behavior. Prefer fixing the earliest correct layer 
rather than patching a downstream symptom. For compatibility changes, examine both the ordinary path and 
boundary cases: empty input, missing fields, type variations, encoding, and exception behavior as 
relevant. Keep public signatures and existing behavior stable outside the issue.

4. Implement a complete,
narrow fix. Use `edit_file` for focused replacements; if matching fails or is ambiguous, reread a smaller
range and retry. Use `write_file` only when a new implementation file is genuinely needed. It is valid to
change runtimePython, Python examples such as `docs_src`, or package metadata when the requested behavior 
requires it. Do not modify tests to satisfy them. Never edit harness-generated `pytest.ini` or 
`conftest.py`,hidden tests, reference patches, or future git history. Avoid broad reformatting and
unrelated cleanup.

5. Verify the actual behavior. First run a small reproduction or a targeted existing 
test tied to the issue. 
Prefer `python3 -c` for a tiny isolated check; use `python3 -m pytest path/to/test.py::test_name -q` 
when a relevant test exists. Do not run the full repository suite, install dependencies,
or use the network. If a check fails, read the failure and repair a regression or incorrect assumption. 
Distinguish unrelated environment errors from behavior failures. For a broad feature, verify at least one 
positive and one boundary case when practical. Keep temporary reproduction scripts in /tmp so they cannot
enter the patch. 

6. Review before submission.
Run `git diff --check`, `git status --short`, and `git diff HEAD` (narrow the diff display if large). 
Ensure every changed file is intentional and that the patch contains implementation, not scratch files.
Call `get_status()` when budget is uncertain. Reserve enough time for this review and the final tool 
action. If time is tight, submit the best defensible working edit; do not abandon a plausible 
fix solely because optional tests cannot run. 7. Call `submit_patch()` exactly once as the final tool
action. It is free and captures git changes, including intended new files. Check its returned patch size
and file count. Then give a concise final text summary of what changed and what was verified.
Tool behavior: commands have a 300-second ceiling and truncated output; file reads are capped. 
Keep commands and edits small. The environment is offline and dependencies are already staged. Use 
tools promptly rather than spending long turns on speculative prose. If a tool call fails, inspect the 
error and take a simpler route. Do not fabricate source content or test outcomes.'

"""



In [4]:
ROR1_FILES = {
  "agent.yaml": ROR1_YAML,
  "configs/sampling.yaml": ROR1_SAMPLING,
  "prompts/system.md": ROR1_PROMPT,
}


def validate_files(files):
  assert "agent.yaml" in files

  allowed = {
    ".yaml", ".yml", ".md", ".txt",
    ".py", ".json", ".safetensors"
  }

  for name, content in files.items():
    path = Path(name)

    assert not path.is_absolute()
    assert ".." not in path.parts
    assert path.suffix in allowed
    assert isinstance(content, str)
    assert content.strip()

  root_text = files["agent.yaml"]

  # Make sure !include targets exist
  for line in root_text.splitlines():
    if "!include" in line:
      included = line.split("!include", 1)[1].strip()

      assert included in files, (
        f"Missing include: {included}"
      )

  # Make YAML parseable for our basic checks
  parsed = yaml.safe_load(
    root_text.replace("!include ", "")
  )

  assert parsed["name"] == "ror-1"
  assert (
    parsed["model"]
    == "gemma-4-31b-it-qat-w4a16-ct"
  )

  assert "submit_patch" in parsed["tools"]

  return parsed

In [5]:
def write_archive(files, output):
  validate_files(files)

  with zipfile.ZipFile(
    output,
    "w",
    compression=zipfile.ZIP_DEFLATED,
  ) as archive:

    for name, content in sorted(files.items()):
      archive.writestr(
        name,
        content.encode("utf-8")
      )

  # Check ZIP isn't corrupted
  with zipfile.ZipFile(output) as archive:
    assert archive.testzip() is None
    assert archive.namelist().count("agent.yaml") == 1

  return output

In [6]:
submission_path = write_archive(
  ROR1_FILES,
  WORKING / "submission.zip"
)

print(
  f"ror-1 submission: {submission_path} "
  f"({submission_path.stat().st_size:,} bytes)"
)

ror-1 submission: /kaggle/working/submission.zip (2,958 bytes)


In [7]:
import zipfile
from pathlib import Path

submission = Path("/kaggle/working/submission.zip")

assert submission.exists(), "submission.zip was not created"

with zipfile.ZipFile(submission) as z:
  print("Files:")
  for name in z.namelist():
    print(" -", name)

  assert z.testzip() is None
  assert sorted(z.namelist()) == sorted([
    "agent.yaml",
    "configs/sampling.yaml",
    "prompts/system.md",
  ])

print(f" Ready: {submission}")
print(f"Size: {submission.stat().st_size:,} bytes")

Files:
 - agent.yaml
 - configs/sampling.yaml
 - prompts/system.md
 Ready: /kaggle/working/submission.zip
Size: 2,958 bytes


In [8]:
from pathlib import Path
import zipfile
import yaml

WORKING = Path("/kaggle/working")

ROR1_FILES = {
  "agent.yaml": ROR1_YAML,
  "configs/sampling.yaml": ROR1_SAMPLING,
  "prompts/system.md": ROR1_PROMPT,
}

submission_path = WORKING / "submission.zip"

with zipfile.ZipFile(
  submission_path,
  "w",
  compression=zipfile.ZIP_DEFLATED
) as archive:
  for name, content in sorted(ROR1_FILES.items()):
    archive.writestr(name, content.encode("utf-8"))

print(f"Created: {submission_path}")
print(f"Size: {submission_path.stat().st_size:,} bytes")

with zipfile.ZipFile(submission_path) as archive:
  print("Contents:")
  for name in archive.namelist():
    print(" -", name)

Created: /kaggle/working/submission.zip
Size: 2,958 bytes
Contents:
 - agent.yaml
 - configs/sampling.yaml
 - prompts/system.md


In [9]:
import math 
def est_compute():
  """
  Estimate compute time per task 

  Threshold:
    Unknown 
  Max Limit:
    Unknown 
  """

  